In [36]:
import random
from datetime import datetime, timedelta
import pandas as pd
import pyodbc

conn_str = 'DRIVER={ODBC Driver 17 for SQL Server};SERVER=DESKTOP-4TCLG8I;DATABASE=university;Trusted_Connection=yes'

last_names = [
    'Baggins', 'Took', 'Brandybuck', 'Gamgee', 'Cotton',
    'Durin', 'Oakenshield', 'Ironfoot', 'Balin', 'Thorin',
    'Stormwind', 'Blackstone', 'Thundershield', 'Frostwhisper', 'Deepstone', 'Ironfist',
    'Starfire', 'Sunblaze', 'Shadowhunter', 'Ravenshadow',
    'Arren', 'Ged', 'Tormer', 'Morwen', 'Tenar', 'Ceredin', 'Duny', 'Yarrow', 'Dragonking', 'Otter',
    'Kelsier', 'Vin','Tindwyl', 'Sazed', 'Marsh', 'Spook', 'Breeze', 'Clubs',
    'Shade', 'Varden', 'Brom', 'Saphira', 'Roran', 'Murtagh', 'Arya', 'Oromis',
    'Pevensie', 'Aslan', 'Tumnus', 'Shasta', 'Reepicheep',
    'Galathor', 'Ravenshadow', 'Stormblade', 'Ironfoot', 'Sunstrike', 'Starfury', 'Dragonsworn',
    'Shadowmoon', 'Tormar', 'Ironfist', 'Blackthorne', 'Everwinter', 'Lightbringer', 'Blackwood',
    'Shadowhunter', 'Redwyne', 'Iceheart', 'Thornfield', 'Nightbloom', 'Frostfall', 'Stonehelm', 'Duskbane', 'Flameborn', 'Moonshade', 'Ashenforge',
    'Stormrend', 'Nightforge', 'Frostborn', 'Grimward', 'Wolfsbane',
    'Ironveil', 'Silverthorn', 'Darkmere', 'Brightflame', 'Thornhelm',
    'Emberlyn', 'Voidwalker', 'Windrider', 'Duskwatch', 'Blazewind'
]

names = pd.read_csv('characters_no_surnames.csv').iloc[:, 0]

In [37]:

conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

num_students = 50

first_names = names

#last_names = pd.read_csv('lotr_surnames.csv', header=None)[0].tolist()

cursor.execute("SELECT MajorID FROM Major")
major_ids = [row[0] for row in cursor.fetchall()]

def generate_random_birth_date_2():
    start_date = datetime(2000, 1, 1)
    end_date = datetime(2008, 1, 1)
    delta = end_date - start_date
    return start_date + timedelta(days=random.randint(0, delta.days))


students = []
emails = set()
used_names = set()

start_years = [2026]

cursor.execute("SELECT FirstName, LastName FROM Student")
used_names = set((row.FirstName, row.LastName) for row in cursor.fetchall())

for _ in range(num_students):
    # Keep generating until we find a unique name combination
    while True:
        first_name = random.choice(first_names)
        last_name = random.choice(last_names)
        full_name = (first_name, last_name)

        if full_name not in used_names:
            used_names.add(full_name)
            break

    # Generate a unique email
    email = f"{first_name.lower()}.{last_name.lower()}{random.randint(1, 99)}@students.middleearth.edu"
    while email in emails:
        email = f"{first_name.lower()}.{last_name.lower()}{random.randint(1, 99)}@students.middleearth.edu"
    emails.add(email)
    DateOfBirth = generate_random_birth_date_2()
    # Assign major and start year
    major_id = random.choice(major_ids)
    start_year = random.choice(start_years)
    start_year_date = f"{start_year}-01-01"

    # Add student to the list
    student = (first_name, last_name, email,DateOfBirth, major_id, start_year_date)
    students.append(student)

# Insert all students into the database
cursor.executemany("""
    INSERT INTO Student (FirstName, LastName, Email,DateofBirth, MajorID, StartYear)
    VALUES (?, ?, ?, ?, ?,?)
""", students)

conn.commit()
cursor.close()
conn.close()

print(f"Inserted {num_students} students")

Inserted 50 students


In [38]:

conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

cursor.execute("SELECT MajorID, Duration FROM Major")
majors = cursor.fetchall()

cursor.execute("SELECT SubjectID FROM Subject")
subject_ids = [row[0] for row in cursor.fetchall()]



subject_major_data = []
subjects_per_semester = 3
for major_id, duration in majors:


    available_subjects = subject_ids.copy()
    random.shuffle(available_subjects)

    if len(available_subjects) < subjects_per_semester:
        print(f"Warning: Not enough available subjects for MajorID {major_id}. Available: {len(available_subjects)}, Required: {subjects_per_semester}")
        selected_subjects = available_subjects
    else:
        selected_subjects = available_subjects[:subjects_per_semester]
    semester=1
    for subject_id in selected_subjects:
        subject_major_data.append((major_id, subject_id, semester))

conn.commit()
print(f"Inserted {len(subject_major_data)} subject-major associations.")

cursor.close()
conn.close()


Inserted 81 subject-major associations.


In [39]:
import random
import pyodbc


conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

cursor.execute("SELECT LecturerID FROM UniversityTeacher")
teacher_ids = [row[0] for row in cursor.fetchall()]

year = 2026
semester=1
classes = []

semester_1_subjects = [entry for entry in subject_major_data if entry[2] == semester]

classes = []

for major_id, subject_id, _ in semester_1_subjects:
    teacher_id = random.choice(teacher_ids)
    method_of_conducting = random.choice(['stationary', 'online', 'hybrid'])

    classes.append((year, subject_id, major_id, semester, teacher_id, method_of_conducting))

# Wstawienie danych do bazy
cursor.executemany("""
    INSERT INTO SubjectInCertainSemester (YearOfConduction, SubjectID, MajorID, Semester, TeacherID, MethodOfConducting)
    VALUES (?, ?, ?, ?, ?, ?)
""", classes)

conn.commit()
cursor.close()
conn.close()

print(f"Inserted {len(classes)} classes into SubjectInCertainSemester table.")
display(classes)

Inserted 81 classes into SubjectInCertainSemester table.


[(2026, 21, 1, 1, 67, 'hybrid'),
 (2026, 1, 1, 1, 94, 'hybrid'),
 (2026, 11, 1, 1, 59, 'hybrid'),
 (2026, 29, 2, 1, 30, 'hybrid'),
 (2026, 34, 2, 1, 100, 'stationary'),
 (2026, 16, 2, 1, 69, 'stationary'),
 (2026, 11, 3, 1, 79, 'online'),
 (2026, 18, 3, 1, 90, 'hybrid'),
 (2026, 2, 3, 1, 70, 'online'),
 (2026, 27, 4, 1, 16, 'stationary'),
 (2026, 15, 4, 1, 53, 'hybrid'),
 (2026, 32, 4, 1, 72, 'online'),
 (2026, 5, 5, 1, 35, 'online'),
 (2026, 24, 5, 1, 22, 'stationary'),
 (2026, 2, 5, 1, 27, 'stationary'),
 (2026, 2, 6, 1, 55, 'hybrid'),
 (2026, 30, 6, 1, 56, 'stationary'),
 (2026, 16, 6, 1, 11, 'online'),
 (2026, 12, 7, 1, 26, 'online'),
 (2026, 24, 7, 1, 60, 'hybrid'),
 (2026, 4, 7, 1, 78, 'online'),
 (2026, 11, 8, 1, 17, 'online'),
 (2026, 22, 8, 1, 14, 'hybrid'),
 (2026, 9, 8, 1, 78, 'online'),
 (2026, 37, 9, 1, 16, 'stationary'),
 (2026, 16, 9, 1, 87, 'stationary'),
 (2026, 29, 9, 1, 52, 'stationary'),
 (2026, 2, 10, 1, 43, 'stationary'),
 (2026, 6, 10, 1, 60, 'stationary'),
 (202

In [40]:
import random
import pyodbc

conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

# Fetch all students with their major
cursor.execute("""
    SELECT StudentID, MajorID FROM Student
    WHERE StartYear = '2026-01-01'
""")
students = cursor.fetchall()

# Fetch only Semester 1 of 2026 classes
cursor.execute("""
    SELECT ClassesID, SubjectID, MajorID FROM SubjectInCertainSemester
    WHERE YearOfConduction = 2026 AND Semester = 1
""")
classes = cursor.fetchall()

# Organize classes by major
major_classes = {}
for class_id, subject_id, major_id in classes:
    major_classes.setdefault(major_id, []).append((class_id, subject_id))

# Prepare grade records
grades_data = []
existing_grades = set()

for student_id, major_id in students:
    if major_id in major_classes:
        for class_id, subject_id in major_classes[major_id]:
            if (student_id, class_id) not in existing_grades:
                grade = random.choice([2, 3, 3.5, 4, 4.5, 5])

                pass_status = int(grade >= 3)
                retake_status = int(grade < 3)

                grades_data.append((student_id, class_id, grade, pass_status, retake_status))
                existing_grades.add((student_id, class_id))

# Insert into Grades table if not already present
for grade_data in grades_data:
    student_id, class_id, grade, pass_status, retake_status = grade_data

    cursor.execute("""
        SELECT COUNT(*) FROM Grades
        WHERE StudentID = ? AND ClassesID = ?
    """, student_id, class_id)

    if cursor.fetchone()[0] == 0:
        cursor.execute("""
            INSERT INTO Grades (StudentID, ClassesID, Grade, Pass, Retaking)
            VALUES (?, ?, ?, ?, ?)
        """, grade_data)

conn.commit()
cursor.close()
conn.close()

print(f"Inserted {len(grades_data)} grades for Semester 1 of 2026.")

Inserted 150 grades for Semester 1 of 2026.


In [41]:
import random
import pandas as pd
import pyodbc
import os

from openpyxl import load_workbook

# Database connection
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

# Fetch only Semester 1 of 2026 classes
cursor.execute("""
    SELECT ClassesID, SubjectID, Semester, YearOfConduction, TeacherID
    FROM SubjectInCertainSemester
    WHERE Semester = 1 AND YearOfConduction = 2026
""")
classes_data = cursor.fetchall()
class_info = {row[0]: (row[1], row[2], row[3], row[4]) for row in classes_data}

# Fetch Grades related to those classes
cursor.execute("""
    SELECT StudentID, ClassesID
    FROM Grades
""")
grades = cursor.fetchall()

# Filter grades to only include relevant class IDs
valid_class_ids = set(class_info.keys())
filtered_grades = [(student_id, class_id) for student_id, class_id in grades if class_id in valid_class_ids]

# Survey options
most_useful_component_options = ["Lectures", "Labs", "Readings", "Assignments"]
workload_perception_options = ["Light", "Moderate", "Heavy"]
yes_no_options = ["Yes", "No"]

# Survey generation
survey_data = []
mean = 8
stddev = 2
survey_id_start = 1

# Prepare file and adjust SurveyID
file_name = "survey_results.xlsx"
if os.path.exists(file_name):
    existing_df = pd.read_excel(file_name)
    survey_id_start = existing_df["SurveyID"].max() + 1
else:
    existing_df = pd.DataFrame()

# Build survey rows for filtered grades
survey_id = survey_id_start
for student_id, class_id in filtered_grades:
    subject_id, semester, year, lecturer_id = class_info[class_id]
    trend = random.randint(0, 2)  # Apply minor trend variation if needed

    survey_entry = [
        survey_id,
        student_id,
        lecturer_id,
        class_id,
        subject_id,
        semester,
        year,
        max(random.randint(1, 10) - trend, 1),
        max(random.randint(1, 10) - trend, 1),
        random.choice(yes_no_options),
        max(random.randint(1, 10) - trend, 1),
        max(random.randint(1, 10) - trend, 1),
        max(random.randint(1, 10) - trend, 1),
        random.choice(most_useful_component_options),
        random.choice(workload_perception_options)
    ]

    survey_data.append(survey_entry)
    survey_id += 1

# Create DataFrame
columns = [
    "SurveyID", "StudentID", "LecturerID", "ClassesID", "SubjectID",
    "Semester", "Year", "Overall_Satisfaction",
    "Usefulness", "Recommend_Course", "Clarity", "Fairness",
    "Use_of_Resources", "Most_Useful_Component", "Workload_Perception"
]

new_df = pd.DataFrame(survey_data, columns=columns)

# Combine and save
final_df = pd.concat([existing_df, new_df], ignore_index=True)
final_df.to_excel(file_name, index=False)

print(f"Appended {len(new_df)} survey entries for Semester 1, 2026 to {file_name}.")

# Close connection
cursor.close()
conn.close()


Appended 150 survey entries for Semester 1, 2026 to survey_results.xlsx.


In [42]:
print()